# Kaggle Clip Generation Pipeline (Audio to PodTok Highlights)

## Mục tiêu Pipeline
Mô hình sẽ nhận vào dữ liệu đã được tiền xử lý ở các bước trên:
- `data/raw_audio_split/*.mp3`
- `data/transcripts/raw_audio_split/*.json`

Pipeline sẽ thực hiện vòng lặp đầy đủ sau:
1. **Load AI Model:** Ở đây sử dụng Google Gemini 1.5 Flash (vì API Free khá dồi dào, ngữ cảnh lên tới 1M token, phù hợp đọc Transcript dài).
2. **Quét dữ liệu:** Quét các tệp Transcript (`.json`).
3. **Prompt Engineering:** Truyền đoạn Text Transcription vào mô hình với Prompt bảo nó "Đóng vai người dựng Short-video để tìm 1 đoạn 60s hay nhất".
4. **Extract mốc thời gian:** Lấy lại Timestamp từ JSON dựa trên Text mà AI chọn.
5. **Cắt Audio (.mp3):** Sử dụng FFmpeg để tự động cắt thành các Clip ngắn (PodTok format).
6. **Lưu Meta Data:** Lưu lại tiêu đề giật gân, tóm tắt và HashTags do AI Gen ra thành File Meta cuối cùng!

## Yêu Cầu Cài Đặt (Kaggle)
1. Cần cài đặt `google-generativeai`.
2. Python `ffmpeg-python` để xử lý âm thanh. 
3. Một API Key từ Google AI Studio (Miễn phí 15 request / phút).

---
**Bắt đầu!**

In [ ]:
%pip install -q orjson pydub ffmpeg-python transformers accelerate bitsandbytes sentencepiece


In [ ]:
import os
import json
import logging
from pathlib import Path
import google.generativeai as genai
from IPython.display import display, Markdown

# ==========================================
# 1. CẤU HÌNH HỆ THỐNG VÀ API KEY
# ==========================================

# Thiết lập logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Thay thế bằng API key thật của bạn (Nên dùng Kaggle Secrets trong thực tế)
GEMINI_API_KEY = "MÃ_API_KEY_CỦA_BẠN"  # <-- ĐIỀN CHÌA KHÓA API VÀO ĐÂY

# ==========================================
# CẤU HÌNH THƯ MỤC TRÊN KAGGLE
# ==========================================
# Bạn có thể trỏ thư mục này xuống dataset của bạn trên Kaggle.
# Giả sử dataset được mount vào /kaggle/input/podtok-data

# Thay đổi đường dẫn này nếu chạy Local thay vì Kaggle
ROOT_DIR = Path('/kaggle/working') # Thư mục xuất dữ liệu trên Kaggle
DATA_DIR = Path('/kaggle/input/podtok-data/data') # Nếu bạn up data dưới dạng Dataset

# Nếu chạy local, bạn mở comment dòng này:
BASE_DIR = Path(os.getcwd()).parent.parent
DATA_DIR = BASE_DIR / "data"
ROOT_DIR = BASE_DIR

# Thư mục chứa File Transcript (.json) và File Audio Split (.mp3)
TRANSCRIPT_DIR = DATA_DIR / "transcripts" / "raw_audio_split"
SPLIT_AUDIO_DIR = DATA_DIR / "raw_audio_split"

# Thư mục đầu ra (Output) cho Clip ngắn 60s
FINAL_CLIPS_DIR = ROOT_DIR / "data" / "final_clips"
FINAL_METADATA_DIR = ROOT_DIR / "data" / "final_metadata"

# Tạo thư mục if not exists
FINAL_CLIPS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_METADATA_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Thư mục Transcript: {TRANSCRIPT_DIR}")
logger.info(f"Thư mục Audio nguồn: {SPLIT_AUDIO_DIR}")
logger.info(f"Thư mục Clip đích: {FINAL_CLIPS_DIR}")

In [ ]:
def read_json(path: Path):
    with open(path, "rb") as f:
        return orjson.loads(f.read())

def write_jsonl(records: List[Dict[str, Any]], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        for row in records:
            f.write(orjson.dumps(row))
            f.write(b"\n")

def normalize_episode_id_from_name(name: str) -> str:
    base = Path(name).stem
    base = base.replace("_split", "")
    return base

def transcript_path_to_episode_id(path: Path) -> str:
    return normalize_episode_id_from_name(path.name)

def audio_path_to_episode_id(path: Path) -> str:
    return normalize_episode_id_from_name(path.name)

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text.strip())
    return text

def looks_like_boundary(text: str) -> bool:
    text = text.strip()
    if not text:
        return False
    if text.endswith((".", "?", "!", ":", ";")):
        return True
    boundary_keywords = [
        "vi du", "tom lai", "nhac lai", "do la", "the nen", "hieu chua",
        "nguyen tac", "bai hoc", "quan trong", "cach hoc", "thay vi"
    ]
    lower = text.lower()
    return any(k in lower for k in boundary_keywords)

def starts_like_bad_opening(text: str) -> bool:
    text = clean_text(text).lower()
    bad_starts = [
        "va ", "thì ", "thi ", "nhưng ", "nhung ", "ví dụ", "vi du",
        "hiểu chưa", "hieu chua", "đó là", "do la", "sẽ ", "se ",
        "hoặc là", "hoac la", "còn ", "con ", "rồi ", "roi "
    ]
    return any(text.startswith(token) for token in bad_starts)

def interval_overlap_ratio(a_start, a_end, b_start, b_end):
    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    if inter <= 0:
        return 0.0
    shorter = min(a_end - a_start, b_end - b_start)
    return inter / max(shorter, 1e-6)

def heuristic_score(text: str, duration_sec: float, start_sec: float) -> Dict[str, Any]:
    text_l = text.lower()
    score = 0.0
    reasons = []

    if IDEAL_MIN_SEC <= duration_sec <= IDEAL_MAX_SEC:
        score += 2.0
        reasons.append("duration_ideal")
    elif MIN_CLIP_SEC <= duration_sec <= MAX_CLIP_SEC:
        score += 1.0
        reasons.append("duration_ok")

    if len(text.split()) >= 50:
        score += 1.5
        reasons.append("enough_words")

    strong_keywords = [
        "vi du", "nguyen tac", "quan trong", "bai hoc", "the nen",
        "thay vi", "cach", "nen", "dung", "micro learning", "pareto"
    ]
    keyword_hits = sum(1 for k in strong_keywords if k in text_l)
    score += min(keyword_hits * 0.6, 2.4)
    if keyword_hits:
        reasons.append(f"keywords_{keyword_hits}")

    filler_keywords = ["a", "um", "uh", "ok", "nhe", "ha", "thi thi"]
    filler_hits = sum(text_l.count(k) for k in filler_keywords)
    if filler_hits >= 5:
        score -= 1.0
        reasons.append("filler_penalty")

    intro_keywords = ["xin chao", "chao mung", "hom nay", "cam on cac ban"]
    if start_sec < 20 and any(k in text_l for k in intro_keywords):
        score -= 2.0
        reasons.append("likely_intro")

    if looks_like_boundary(text):
        score += 1.0
        reasons.append("good_boundary")

    return {
        "heuristic_score": round(score, 3),
        "heuristic_reasons": reasons
    }

def chunk_list(items: List[Any], batch_size: int) -> List[List[Any]]:
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def extract_first_json_block(text: str) -> Dict[str, Any]:
    start = text.find("{")
    if start < 0:
        raise ValueError("No JSON block found")

    depth = 0
    in_string = False
    escaped = False

    for idx in range(start, len(text)):
        ch = text[idx]
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:idx + 1])

    raise ValueError("No complete JSON block found")


def clean_generation_text(text: str) -> str:
    text = (text or "").strip()
    text = text.replace("```json", "```").replace("```JSON", "```")
    if "```" in text:
        parts = text.split("```")
        fenced_parts = [part.strip() for part in parts if part.strip()]
        if fenced_parts:
            text = fenced_parts[0]
    return text.strip()


def make_llm_system_prompt() -> str:
    return """
You are an expert short-form podcast editor.
Return VALID JSON only.
Do not output markdown.
Do not output explanations outside JSON.
Use only the provided candidate IDs.
Choose the strongest clips only.
Write reasons in short English only.
""".strip()


def make_llm_group_prompt(rows: List[Dict[str, Any]], title: str = "", host: str = "", keyword: str = "") -> str:
    title = clean_text(title or "")
    host = clean_text(host or "")
    keyword = clean_text(keyword or "")

    clip_lines = []
    for row in rows:
        candidate_id = row.get("clip_candidate_id") or row.get("clip_id") or "unknown_candidate"
        clip_lines.append(
            f"- clip_candidate_id: {candidate_id}\n"
            f"  time: {round(row['start_sec'], 2)} -> {round(row['end_sec'], 2)}\n"
            f"  heuristic_score: {row['heuristic_score']}\n"
            f"  transcript: {clean_text(row['text'])}"
        )

    clips_text = "\n\n".join(clip_lines)
    top_k = min(LLM_TOP_PER_GROUP, len(rows))

    return f"""
Task:
Select the best short podcast clips from the SAME Vietnamese episode.
Compare candidates against each other and choose only the strongest ones.

Episode metadata:
- title: {title}
- host: {host}
- keyword: {keyword}

Selection rules:
1. The clip must be understandable on its own.
2. The clip should match the episode topic.
3. The clip should give value to a new listener.
4. Prefer clips with hook, insight, example, lesson, or memorable point.
5. Avoid intro, outro, ads, weak transition, noisy transcript, or repeated idea.
6. Avoid picking two candidates with nearly the same meaning.

Important output rules:
- Return VALID JSON only.
- No markdown.
- No extra text.
- Use candidate IDs exactly as provided.
- `top_candidate_ids` must contain exactly {top_k} ids if possible.
- `group_reason` must be a short English phrase.
- `top_reasons` must contain short English phrases.

Return JSON in this exact format:
{{
  "top_candidate_ids": ["...", "..."],
  "group_reason": "short English reason",
  "top_reasons": {{
    "clip_candidate_id_1": "short English reason",
    "clip_candidate_id_2": "short English reason"
  }}
}}

Candidate list:
{clips_text}
""".strip()


def parse_llm_group_output(text: str, valid_candidate_ids: List[str], top_k: int) -> Dict[str, Any]:
    cleaned = clean_generation_text(text)
    valid_set = set(valid_candidate_ids)
    top_ids: List[str] = []
    top_reasons: Dict[str, str] = {}
    group_reason = ""

    parse_attempts = [cleaned]
    try:
        parse_attempts.append(json.dumps(json.loads(cleaned), ensure_ascii=False))
    except Exception:
        pass

    for attempt in parse_attempts:
        try:
            parsed = extract_first_json_block(attempt)
            raw_top_ids = parsed.get("top_candidate_ids", [])
            if isinstance(raw_top_ids, list):
                for candidate_id in raw_top_ids:
                    candidate_id = str(candidate_id).strip()
                    if candidate_id in valid_set and candidate_id not in top_ids:
                        top_ids.append(candidate_id)

            raw_top_reasons = parsed.get("top_reasons", {})
            if isinstance(raw_top_reasons, dict):
                for candidate_id, reason in raw_top_reasons.items():
                    candidate_id = str(candidate_id).strip()
                    if candidate_id in valid_set:
                        top_reasons[candidate_id] = clean_text(str(reason))[:120]

            group_reason = clean_text(str(parsed.get("group_reason", "")))[:160]
            break
        except Exception:
            continue

    if not top_ids:
        ordered_hits = []
        for candidate_id in valid_candidate_ids:
            pos = cleaned.find(candidate_id)
            if pos >= 0:
                ordered_hits.append((pos, candidate_id))
        ordered_hits.sort()
        for _, candidate_id in ordered_hits[:top_k]:
            if candidate_id not in top_ids:
                top_ids.append(candidate_id)
        if top_ids and not group_reason:
            group_reason = "regex_salvaged"

    top_ids = top_ids[:top_k]
    if not group_reason:
        group_reason = "parse_failed" if not top_ids else "selected_top_ids"

    return {
        "top_candidate_ids": top_ids,
        "top_reasons": top_reasons,
        "group_reason": group_reason,
    }


In [ ]:
metadata_rows = read_json(METADATA_PATH)
metadata_map = {row["_id"]: row for row in metadata_rows}

transcript_paths = sorted(TRANSCRIPT_DIR.glob("*.json"))
audio_paths = sorted(SPLIT_AUDIO_DIR.glob("*.mp3"))
audio_map = {audio_path_to_episode_id(p): p for p in audio_paths}

episode_index = []
for tpath in transcript_paths:
    episode_id = transcript_path_to_episode_id(tpath)
    meta = metadata_map.get(episode_id, {})
    row = {
        "episode_id": episode_id,
        "transcript_path": str(tpath),
        "split_audio_path": str(audio_map.get(episode_id, "")),
        "title": meta.get("title"),
        "host": meta.get("host"),
        "keyword": meta.get("keyword"),
        "source_url": meta.get("source_url"),
        "audio_file": meta.get("audio_file")
    }
    episode_index.append(row)

episode_index_path = OUTPUT_DIR / "episode_index.jsonl"
write_jsonl(episode_index, episode_index_path)
print("episodes:", len(episode_index))
print("saved:", episode_index_path)
pd.DataFrame(episode_index).head(3)


In [ ]:
t0 = time.time()

def build_candidates_for_transcript(episode_id: str, transcript: Dict[str, Any]) -> List[Dict[str, Any]]:
    segments = transcript.get("segments", [])
    candidates = []
    n = len(segments)

    for i in range(n):
        start_sec = float(segments[i]["start"])
        parts = []
        last_end = start_sec

        for j in range(i, n):
            seg = segments[j]
            parts.append(clean_text(seg["text"]))
            last_end = float(seg["end"])
            duration = last_end - start_sec

            if duration < MIN_CLIP_SEC:
                continue

            merged_text = clean_text(" ".join(parts))
            is_boundary = looks_like_boundary(seg["text"]) or looks_like_boundary(merged_text)
            bad_opening = starts_like_bad_opening(merged_text)
            is_sentence_complete = bool(is_boundary and not bad_opening)

            if duration <= IDEAL_MAX_SEC and is_boundary:
                candidates.append({
                    "clip_candidate_id": f"{episode_id}_{i:04d}_{j:04d}",
                    "episode_id": episode_id,
                    "start_sec": round(start_sec, 2),
                    "end_sec": round(last_end, 2),
                    "duration_sec": round(duration, 2),
                    "start_segment_idx": i,
                    "end_segment_idx": j,
                    "num_segments": j - i + 1,
                    "text": merged_text,
                    "is_sentence_complete": is_sentence_complete
                })
                break

            if IDEAL_MAX_SEC < duration <= MAX_CLIP_SEC:
                candidates.append({
                    "clip_candidate_id": f"{episode_id}_{i:04d}_{j:04d}",
                    "episode_id": episode_id,
                    "start_sec": round(start_sec, 2),
                    "end_sec": round(last_end, 2),
                    "duration_sec": round(duration, 2),
                    "start_segment_idx": i,
                    "end_segment_idx": j,
                    "num_segments": j - i + 1,
                    "text": merged_text,
                    "is_sentence_complete": is_sentence_complete
                })
                break

            if duration > MAX_CLIP_SEC:
                break

    dedup = []
    seen = set()
    for row in candidates:
        key = (row["start_sec"], row["end_sec"], row["text"][:80])
        if key not in seen:
            seen.add(key)
            dedup.append(row)
    return dedup

all_candidates = []
for tpath in tqdm(transcript_paths, desc="build_candidates"):
    episode_id = transcript_path_to_episode_id(tpath)
    transcript = read_json(tpath)
    episode_meta = metadata_map.get(episode_id, {})
    rows = build_candidates_for_transcript(episode_id, transcript)
    for row in rows:
        row["title"] = episode_meta.get("title")
        row["host"] = episode_meta.get("host")
        row["keyword"] = episode_meta.get("keyword")
        row["source_url"] = episode_meta.get("source_url")
        all_candidates.append(row)

clip_candidates_path = OUTPUT_DIR / "clip_candidates.jsonl"
write_jsonl(all_candidates, clip_candidates_path)
RUNTIME_STATS["build_candidates_sec"] = round(time.time() - t0, 2)
print("total_candidates:", len(all_candidates))
print("saved:", clip_candidates_path)
print("elapsed_sec:", RUNTIME_STATS["build_candidates_sec"])
pd.DataFrame(all_candidates).head(5)


In [ ]:
t0 = time.time()

heuristic_rows = []
for row in tqdm(all_candidates, desc="heuristic_score"):
    score_info = heuristic_score(row["text"], row["duration_sec"], row["start_sec"])
    new_row = dict(row)
    new_row.update(score_info)
    heuristic_rows.append(new_row)

heuristic_df = pd.DataFrame(heuristic_rows)
heuristic_df = heuristic_df.sort_values(["episode_id", "heuristic_score"], ascending=[True, False])
heuristic_top_df = heuristic_df.groupby("episode_id").head(MAX_CANDIDATES_PER_EPISODE).reset_index(drop=True)
heuristic_top_rows = heuristic_top_df.to_dict(orient="records")

clip_candidates_heuristic_path = OUTPUT_DIR / "clip_candidates_heuristic.jsonl"
write_jsonl(heuristic_top_rows, clip_candidates_heuristic_path)
RUNTIME_STATS["heuristic_score_sec"] = round(time.time() - t0, 2)
print("top_candidates_after_heuristic:", len(heuristic_top_rows))
print("saved:", clip_candidates_heuristic_path)
print("elapsed_sec:", RUNTIME_STATS["heuristic_score_sec"])
heuristic_top_df.head(5)


In [ ]:
t0 = time.time()
scored_rows = []

if USE_LLM:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, pipeline

    llm_load_failed = False
    llm_load_error = None

    try:
        tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"

        model_kwargs = {
            "device_map": "auto",
            "trust_remote_code": True
        }
        if USE_4BIT:
            model_kwargs["load_in_4bit"] = True
            model_kwargs["dtype"] = torch.float16
        else:
            model_kwargs["dtype"] = torch.bfloat16 if USE_BFLOAT16 else torch.float16
            if USE_FLASH_ATTENTION:
                model_kwargs["attn_implementation"] = "sdpa"

        model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, **model_kwargs)

        generation_config = GenerationConfig(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        generator = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            batch_size=LLM_BATCH_SIZE,
            return_full_text=False
        )
    except Exception as e:
        llm_load_failed = True
        llm_load_error = str(e)
        print("LLM load failed, fallback to heuristic-only:", llm_load_error)

    if not llm_load_failed:
        llm_groups = []
        for episode_id, group_df in heuristic_top_df.groupby("episode_id", sort=False):
            episode_rows = group_df.sort_values("heuristic_score", ascending=False).to_dict(orient="records")
            for group_idx, row_batch in enumerate(chunk_list(episode_rows, LLM_GROUP_SIZE)):
                messages = [
                    {"role": "system", "content": make_llm_system_prompt()},
                    {
                        "role": "user",
                        "content": make_llm_group_prompt(
                            row_batch,
                            title=row_batch[0].get("title", ""),
                            host=row_batch[0].get("host", ""),
                            keyword=row_batch[0].get("keyword", "")
                        )
                    }
                ]
                rendered_prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
                llm_groups.append({
                    "episode_id": episode_id,
                    "group_idx": group_idx,
                    "rows": row_batch,
                    "prompt": rendered_prompt
                })

        temp_rows = []
        for prompt_batch in tqdm(list(chunk_list(llm_groups, LLM_BATCH_SIZE)), desc="llm_rank_groups"):
            prompts = [item["prompt"] for item in prompt_batch]
            outputs = generator(prompts, generation_config=generation_config)

            for group_info, out in zip(prompt_batch, outputs):
                generated_text = out[0]["generated_text"] if isinstance(out, list) else out["generated_text"]
                candidate_ids = [row.get("clip_candidate_id") or row.get("clip_id") or "" for row in group_info["rows"]]
                parsed = parse_llm_group_output(generated_text, candidate_ids, min(LLM_TOP_PER_GROUP, len(candidate_ids)))
                top_ids = parsed.get("top_candidate_ids", [])
                top_reasons = parsed.get("top_reasons", {})
                group_reason = parsed.get("group_reason", "")

                for row in group_info["rows"]:
                    candidate_id = row.get("clip_candidate_id") or row.get("clip_id") or ""
                    heuristic_score = float(row["heuristic_score"])
                    keep_priority = 1 if candidate_id in top_ids else 0

                    if candidate_id in top_ids:
                        rank_idx = top_ids.index(candidate_id)
                        parsed_score = heuristic_score + 1.8 - (0.35 * rank_idx)
                        llm_label = "keep"
                        llm_reason = top_reasons.get(candidate_id, group_reason)
                    else:
                        parsed_score = heuristic_score - 0.15
                        llm_label = "fallback" if group_reason == "parse_failed" else "consider"
                        llm_reason = "parse_failed" if group_reason == "parse_failed" else group_reason

                    new_row = dict(row)
                    new_row["llm_raw_score"] = round(parsed_score, 3)
                    new_row["llm_score"] = round(parsed_score + (0.5 if keep_priority else 0.0), 3)
                    new_row["llm_label"] = llm_label
                    new_row["llm_reason"] = (llm_reason or "")[:300]
                    new_row["llm_keep_priority"] = keep_priority
                    new_row["llm_group_idx"] = group_info["group_idx"]
                    temp_rows.append(new_row)

        if temp_rows:
            temp_df = pd.DataFrame(temp_rows)
            temp_df = temp_df.sort_values(
                ["episode_id", "llm_keep_priority", "llm_score", "heuristic_score"],
                ascending=[True, False, False, False]
            )
            scored_rows = temp_df.to_dict(orient="records")

    if llm_load_failed or not scored_rows:
        for row in heuristic_top_rows:
            new_row = dict(row)
            new_row["llm_raw_score"] = row["heuristic_score"]
            new_row["llm_score"] = row["heuristic_score"]
            new_row["llm_label"] = "heuristic_fallback"
            new_row["llm_reason"] = llm_load_error[:300] if llm_load_error else "llm_unavailable"
            new_row["llm_keep_priority"] = 0
            new_row["llm_group_idx"] = -1
            scored_rows.append(new_row)
else:
    for row in heuristic_top_rows:
        new_row = dict(row)
        new_row["llm_raw_score"] = row["heuristic_score"]
        new_row["llm_score"] = row["heuristic_score"]
        new_row["llm_label"] = "heuristic_only"
        new_row["llm_reason"] = ",".join(row.get("heuristic_reasons", []))
        new_row["llm_keep_priority"] = 0
        new_row["llm_group_idx"] = -1
        scored_rows.append(new_row)

clip_candidates_llm_scored_path = OUTPUT_DIR / "clip_candidates_llm_scored.jsonl"
write_jsonl(scored_rows, clip_candidates_llm_scored_path)
RUNTIME_STATS["llm_score_sec"] = round(time.time() - t0, 2)
print("saved:", clip_candidates_llm_scored_path)
print("elapsed_sec:", RUNTIME_STATS["llm_score_sec"])
pd.DataFrame(scored_rows).sort_values(["llm_keep_priority", "llm_score"], ascending=[False, False]).head(5)


In [ ]:
def select_non_overlapping(rows: List[Dict[str, Any]], top_k: int) -> List[Dict[str, Any]]:
    rows = sorted(
        rows,
        key=lambda x: (
            x.get("llm_keep_priority", 0),
            x["llm_score"],
            x["heuristic_score"]
        ),
        reverse=True
    )
    chosen = []
    for row in rows:
        conflict = False
        for existing in chosen:
            overlap = interval_overlap_ratio(
                row["start_sec"], row["end_sec"],
                existing["start_sec"], existing["end_sec"]
            )
            same_near_segments = abs(row["start_segment_idx"] - existing["start_segment_idx"]) <= 5 or abs(row["end_segment_idx"] - existing["end_segment_idx"]) <= 5
            same_prefix = clean_text(row["text"])[:120].lower() == clean_text(existing["text"])[:120].lower()
            if overlap > 0.30 or same_near_segments or same_prefix:
                conflict = True
                break
        if not conflict:
            chosen.append(row)
        if len(chosen) >= top_k:
            break
    return chosen

final_rows = []
score_df = pd.DataFrame(scored_rows)
for episode_id, group_df in score_df.groupby("episode_id"):
    selected = select_non_overlapping(group_df.to_dict(orient="records"), TOP_K_CLIPS_PER_EPISODE)
    for idx, row in enumerate(selected, start=1):
        new_row = dict(row)
        new_row["clip_id"] = f"clip_{episode_id}_{idx:04d}"
        new_row["segment_index"] = idx
        final_rows.append(new_row)

final_clip_selection_path = OUTPUT_DIR / "final_clip_selection.jsonl"
write_jsonl(final_rows, final_clip_selection_path)
print("final_clips:", len(final_rows))
print("saved:", final_clip_selection_path)
pd.DataFrame(final_rows).head(8)


In [ ]:
t0 = time.time()
item_metadata_rows = []
final_df = pd.DataFrame(final_rows)

for episode_id, group_df in tqdm(final_df.groupby("episode_id"), desc="export_clips_by_episode"):
    audio_path = audio_map.get(episode_id)
    if audio_path is None or not Path(audio_path).exists():
        continue

    audio = AudioSegment.from_file(audio_path)
    clip_subdir = CLIPS_DIR / episode_id
    clip_subdir.mkdir(parents=True, exist_ok=True)

    for row in group_df.to_dict(orient="records"):
        clip_filename = f"{row['clip_id']}.mp3"
        clip_path = clip_subdir / clip_filename
        relative_clip_path = Path("outputs") / "clips" / episode_id / clip_filename
        start_ms = int(row["start_sec"] * 1000)
        end_ms = int(row["end_sec"] * 1000)
        clip_audio = audio[start_ms:end_ms]
        clip_audio.export(clip_path, format="mp3")

        item_metadata_rows.append({
            "clip_id": row["clip_id"],
            "episode_id": episode_id,
            "title": row.get("title"),
            "host": row.get("host"),
            "keyword": row.get("keyword"),
            "source_url": row.get("source_url"),
            "source_audio_file": Path(audio_path).name,
            "clip_audio_path": str(relative_clip_path),
            "start_sec": row["start_sec"],
            "end_sec": row["end_sec"],
            "duration_sec": row["duration_sec"],
            "start_segment_idx": row["start_segment_idx"],
            "end_segment_idx": row["end_segment_idx"],
            "num_segments": row["num_segments"],
            "transcript_text": row["text"],
            "is_sentence_complete": row["is_sentence_complete"],
            "heuristic_score": row["heuristic_score"],
            "heuristic_reasons": row["heuristic_reasons"],
            "llm_score": row["llm_score"],
            "llm_label": row["llm_label"],
            "llm_reason": row["llm_reason"]
        })

item_metadata_path = OUTPUT_DIR / "item_metadata.jsonl"
item_metadata_csv_path = OUTPUT_DIR / "item_metadata.csv"
write_jsonl(item_metadata_rows, item_metadata_path)
pd.DataFrame(item_metadata_rows).to_csv(item_metadata_csv_path, index=False)
RUNTIME_STATS["export_clips_sec"] = round(time.time() - t0, 2)

print("item_metadata_rows:", len(item_metadata_rows))
print("saved:", item_metadata_path)
print("saved:", item_metadata_csv_path)
print("elapsed_sec:", RUNTIME_STATS["export_clips_sec"])
pd.DataFrame(item_metadata_rows).head(5)


In [ ]:
runtime_report = {
    "episodes": len(episode_index),
    "all_candidates": len(all_candidates),
    "heuristic_top_candidates": len(heuristic_top_rows),
    "final_clips": len(final_rows),
    "exported_item_metadata": len(item_metadata_rows),
    **RUNTIME_STATS
}

runtime_report_path = OUTPUT_DIR / "runtime_report.json"
with open(runtime_report_path, "wb") as f:
    f.write(orjson.dumps(runtime_report, option=orjson.OPT_INDENT_2))

print(json.dumps(runtime_report, indent=2, ensure_ascii=False))
print("saved:", runtime_report_path)


## Kiem tra nhanh sau khi chay

Ban nen check 4 thu nay:

1. `outputs/item_metadata.csv` co du clip chua.
2. `outputs/clips/...` co tao MP3 that hay khong.
3. Nghe thu 5-10 clip xem noi dung co tron y khong.
4. Neu clip chua hay, dieu chinh:
   - heuristic keywords
   - `MIN_CLIP_SEC`, `MAX_CLIP_SEC`
   - bat `USE_LLM = True`

## Buoc tiep theo sau notebook nay

Khi da co `item_metadata.jsonl`, ban co the lam tiep:
- text embedding
- audio embedding
- vector fusion
- vector DB
- mock interactions
- train SASRec


## Double Check bang LLM

Cell duoi day dung LLM de check lai tung clip da duoc chon o buoc cuoi.

Muc tieu:
- bat clip intro / outro / quang cao
- bat clip bi dut y hoac transcript qua ban
- danh gia xem clip co dang giu de lam short hay khong

Output:
- `outputs/final_clip_qa.jsonl`
- `outputs/item_metadata_qa_reviewed.jsonl`
- `outputs/item_metadata_qa_passed.jsonl`


In [ ]:
t0 = time.time()

QA_MAX_NEW_TOKENS = 96
qa_rows = []


def make_final_qa_prompt(row: Dict[str, Any]) -> str:
    return f"""
You are checking a final selected short podcast clip.
Return VALID JSON only.
Do not output markdown.
Do not output any text outside JSON.
Write `reason` in short English only.

Decide if this clip should be kept as a final short clip.
Reject clips that are intro, outro, ad, broken, noisy, incomplete, or weak.

Episode metadata:
- title: {clean_text(row.get('title', ''))}
- host: {clean_text(row.get('host', ''))}
- keyword: {clean_text(row.get('keyword', ''))}

Clip info:
- clip_id: {row.get('clip_id', '')}
- duration_sec: {row.get('duration_sec', 0)}
- is_sentence_complete: {row.get('is_sentence_complete', False)}
- heuristic_score: {row.get('heuristic_score', 0)}
- llm_score: {row.get('llm_score', 0)}
- transcript: {clean_text(row.get('transcript_text', ''))}

Return JSON in this exact format:
{{
  "qa_label": "keep" or "reject",
  "qa_score": 0 to 10,
  "reason": "short English reason",
  "flags": {{
    "intro": true or false,
    "outro": true or false,
    "ad": true or false,
    "broken": true or false,
    "noisy": true or false,
    "weak": true or false
  }}
}}
""".strip()


def parse_final_qa_output(text: str) -> Dict[str, Any]:
    cleaned = clean_generation_text(text)
    try:
        parsed = extract_first_json_block(cleaned)
        flags = parsed.get("flags", {}) if isinstance(parsed.get("flags", {}), dict) else {}
        qa_label = str(parsed.get("qa_label", "keep")).strip().lower()
        if qa_label not in {"keep", "reject"}:
            qa_label = "keep"
        return {
            "qa_label": qa_label,
            "qa_score": round(float(parsed.get("qa_score", 6.0)), 3),
            "qa_reason": clean_text(str(parsed.get("reason", "")))[:200],
            "qa_flag_intro": bool(flags.get("intro", False)),
            "qa_flag_outro": bool(flags.get("outro", False)),
            "qa_flag_ad": bool(flags.get("ad", False)),
            "qa_flag_broken": bool(flags.get("broken", False)),
            "qa_flag_noisy": bool(flags.get("noisy", False)),
            "qa_flag_weak": bool(flags.get("weak", False)),
            "qa_parse_status": "parsed"
        }
    except Exception:
        text_l = cleaned.lower()
        return {
            "qa_label": "reject" if any(token in text_l for token in ["intro", "outro", "advert", "ad", "broken", "weak", "noisy"]) else "keep",
            "qa_score": 4.5,
            "qa_reason": "parse_failed",
            "qa_flag_intro": "intro" in text_l,
            "qa_flag_outro": "outro" in text_l,
            "qa_flag_ad": "advert" in text_l or " ad " in f" {text_l} ",
            "qa_flag_broken": "broken" in text_l,
            "qa_flag_noisy": "noisy" in text_l,
            "qa_flag_weak": "weak" in text_l,
            "qa_parse_status": "parse_failed"
        }


qa_generator = None
qa_tokenizer = None
qa_generation_config = None
qa_unavailable_reason = ""

if not USE_LLM:
    qa_unavailable_reason = "USE_LLM_false"
else:
    if "generator" in globals() and generator is not None and "tokenizer" in globals() and tokenizer is not None:
        qa_generator = generator
        qa_tokenizer = tokenizer
        qa_generation_config = GenerationConfig(
            max_new_tokens=QA_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    else:
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, pipeline

            qa_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
            if qa_tokenizer.pad_token is None:
                qa_tokenizer.pad_token = qa_tokenizer.eos_token
            qa_tokenizer.padding_side = "left"

            qa_model_kwargs = {
                "device_map": "auto",
                "trust_remote_code": True
            }
            if USE_4BIT:
                qa_model_kwargs["load_in_4bit"] = True
                qa_model_kwargs["dtype"] = torch.float16
            else:
                qa_model_kwargs["dtype"] = torch.bfloat16 if USE_BFLOAT16 else torch.float16
                if USE_FLASH_ATTENTION:
                    qa_model_kwargs["attn_implementation"] = "sdpa"

            qa_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, **qa_model_kwargs)
            qa_generation_config = GenerationConfig(
                max_new_tokens=QA_MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=qa_tokenizer.pad_token_id,
                eos_token_id=qa_tokenizer.eos_token_id,
            )
            qa_generator = pipeline(
                "text-generation",
                model=qa_model,
                tokenizer=qa_tokenizer,
                batch_size=LLM_BATCH_SIZE,
                return_full_text=False
            )
        except Exception as e:
            qa_unavailable_reason = str(e)

if qa_generator is None:
    for row in item_metadata_rows:
        new_row = dict(row)
        new_row.update({
            "qa_label": "keep",
            "qa_score": row.get("llm_score", row.get("heuristic_score", 0)),
            "qa_reason": qa_unavailable_reason[:200] if qa_unavailable_reason else "qa_unavailable",
            "qa_flag_intro": False,
            "qa_flag_outro": False,
            "qa_flag_ad": False,
            "qa_flag_broken": False,
            "qa_flag_noisy": False,
            "qa_flag_weak": False,
            "qa_parse_status": "qa_skipped"
        })
        qa_rows.append(new_row)
else:
    qa_groups = []
    for row in item_metadata_rows:
        messages = [
            {"role": "system", "content": "You are a strict final QA reviewer for short podcast clips. Return JSON only."},
            {"role": "user", "content": make_final_qa_prompt(row)}
        ]
        rendered_prompt = qa_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        qa_groups.append({"row": row, "prompt": rendered_prompt})

    for prompt_batch in tqdm(list(chunk_list(qa_groups, LLM_BATCH_SIZE)), desc="final_clip_qa"):
        prompts = [item["prompt"] for item in prompt_batch]
        outputs = qa_generator(prompts, generation_config=qa_generation_config)
        for item, out in zip(prompt_batch, outputs):
            generated_text = out[0]["generated_text"] if isinstance(out, list) else out["generated_text"]
            qa_info = parse_final_qa_output(generated_text)
            new_row = dict(item["row"])
            new_row.update(qa_info)
            qa_rows.append(new_row)

final_clip_qa_path = OUTPUT_DIR / "final_clip_qa.jsonl"
write_jsonl(qa_rows, final_clip_qa_path)
RUNTIME_STATS["final_clip_qa_sec"] = round(time.time() - t0, 2)

print("qa_rows:", len(qa_rows))
print("saved:", final_clip_qa_path)
print("elapsed_sec:", RUNTIME_STATS["final_clip_qa_sec"])
pd.DataFrame(qa_rows)[["clip_id", "qa_label", "qa_score", "qa_reason", "qa_parse_status"]].head(10)


In [ ]:
qa_reviewed_path = OUTPUT_DIR / "item_metadata_qa_reviewed.jsonl"
qa_passed_path = OUTPUT_DIR / "item_metadata_qa_passed.jsonl"
qa_summary_path = OUTPUT_DIR / "final_clip_qa_summary.json"

qa_reviewed_rows = list(qa_rows)
qa_passed_rows = []
for row in qa_reviewed_rows:
    reject_by_label = row.get("qa_label") == "reject"
    reject_by_flags = any([
        row.get("qa_flag_intro", False),
        row.get("qa_flag_outro", False),
        row.get("qa_flag_ad", False),
        row.get("qa_flag_broken", False),
    ])
    reject_by_score = float(row.get("qa_score", 0.0)) < 5.5

    if not (reject_by_label or reject_by_flags or reject_by_score):
        qa_passed_rows.append(row)

write_jsonl(qa_reviewed_rows, qa_reviewed_path)
write_jsonl(qa_passed_rows, qa_passed_path)

qa_summary = {
    "qa_reviewed": len(qa_reviewed_rows),
    "qa_passed": len(qa_passed_rows),
    "qa_rejected": len(qa_reviewed_rows) - len(qa_passed_rows),
    "qa_keep_rate": round(len(qa_passed_rows) / max(len(qa_reviewed_rows), 1), 3),
    "qa_parse_failed": sum(1 for row in qa_reviewed_rows if row.get("qa_parse_status") == "parse_failed"),
    "qa_flag_intro": sum(1 for row in qa_reviewed_rows if row.get("qa_flag_intro", False)),
    "qa_flag_outro": sum(1 for row in qa_reviewed_rows if row.get("qa_flag_outro", False)),
    "qa_flag_ad": sum(1 for row in qa_reviewed_rows if row.get("qa_flag_ad", False)),
    "qa_flag_broken": sum(1 for row in qa_reviewed_rows if row.get("qa_flag_broken", False)),
    "qa_flag_noisy": sum(1 for row in qa_reviewed_rows if row.get("qa_flag_noisy", False)),
    "qa_flag_weak": sum(1 for row in qa_reviewed_rows if row.get("qa_flag_weak", False)),
}

with open(qa_summary_path, "wb") as f:
    f.write(orjson.dumps(qa_summary, option=orjson.OPT_INDENT_2))

print("saved:", qa_reviewed_path)
print("saved:", qa_passed_path)
print("saved:", qa_summary_path)
print(json.dumps(qa_summary, indent=2, ensure_ascii=False))
pd.DataFrame(qa_passed_rows)[["clip_id", "qa_label", "qa_score", "qa_reason"]].head(10)


In [ ]:
qa_passed_export_path = OUTPUT_DIR / "item_metadata_after_qa.jsonl"
qa_passed_export_csv_path = OUTPUT_DIR / "item_metadata_after_qa.csv"
qa_passed_summary_path = OUTPUT_DIR / "after_qa_summary.json"

qa_passed_item_rows = []
for row in qa_passed_rows:
    qa_row = dict(row)
    qa_row["final_status"] = "approved_after_qa"
    qa_passed_item_rows.append(qa_row)

write_jsonl(qa_passed_item_rows, qa_passed_export_path)
pd.DataFrame(qa_passed_item_rows).to_csv(qa_passed_export_csv_path, index=False)

qa_passed_summary = {
    "episodes_total": len(episode_index),
    "clips_before_qa": len(item_metadata_rows),
    "clips_after_qa": len(qa_passed_item_rows),
    "qa_rejected": len(item_metadata_rows) - len(qa_passed_item_rows),
    "qa_keep_rate": round(len(qa_passed_item_rows) / max(len(item_metadata_rows), 1), 3),
    "avg_llm_score_after_qa": round(sum(float(row.get("llm_score", 0.0)) for row in qa_passed_item_rows) / max(len(qa_passed_item_rows), 1), 3),
    "avg_qa_score_after_qa": round(sum(float(row.get("qa_score", 0.0)) for row in qa_passed_item_rows) / max(len(qa_passed_item_rows), 1), 3),
}

with open(qa_passed_summary_path, "wb") as f:
    f.write(orjson.dumps(qa_passed_summary, option=orjson.OPT_INDENT_2))

print("saved:", qa_passed_export_path)
print("saved:", qa_passed_export_csv_path)
print("saved:", qa_passed_summary_path)
print(json.dumps(qa_passed_summary, indent=2, ensure_ascii=False))
pd.DataFrame(qa_passed_item_rows)[["clip_id", "episode_id", "llm_score", "qa_score", "final_status"]].head(10)


## Dedup Truoc Final Export

Cell duoi day dedup tren toan bo bo clip sau QA.

Muc tieu:
- loai clip transcript trung hoan toan
- loai clip gan nhu trung nhau giua cac episode khac nhau
- uu tien giu clip co `qa_score`, `llm_score`, `heuristic_score` cao hon

Output:
- `outputs/final_dedup_review.jsonl`
- `outputs/final_dedup_summary.json`
- bien `final_deduped_rows` de export o cell final cuoi cung


In [ ]:
import difflib


def normalize_transcript_for_dedup(text: str) -> str:
    text = clean_text(text or "").lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def transcript_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    return difflib.SequenceMatcher(None, a, b).ratio()


def dedup_rank_key(row: Dict[str, Any]):
    duration = float(row.get("duration_sec", 0.0))
    duration_closeness = -abs(duration - 35.0)
    return (
        float(row.get("qa_score", 0.0)),
        float(row.get("llm_score", 0.0)),
        float(row.get("heuristic_score", 0.0)),
        duration_closeness,
        int(bool(row.get("is_sentence_complete", False))),
    )


sorted_final_rows = sorted(qa_passed_item_rows, key=dedup_rank_key, reverse=True)
kept_rows = []
dedup_review_rows = []

for row in sorted_final_rows:
    current = dict(row)
    current_norm_text = normalize_transcript_for_dedup(current.get("transcript_text", ""))
    current_prefix = current_norm_text[:220]
    current_source = clean_text(current.get("source_url", ""))

    duplicate_of = ""
    duplicate_reason = ""
    duplicate_similarity = 0.0

    for kept in kept_rows:
        kept_norm_text = kept.get("_dedup_norm_text", "")
        kept_prefix = kept_norm_text[:220]
        kept_source = clean_text(kept.get("source_url", ""))

        same_exact_text = current_norm_text and current_norm_text == kept_norm_text
        same_prefix = current_prefix and current_prefix == kept_prefix
        same_source = current_source and kept_source and current_source == kept_source
        similarity = transcript_similarity(current_norm_text, kept_norm_text)
        highly_similar = similarity >= 0.97
        similar_same_source = same_source and similarity >= 0.92

        if same_exact_text or same_prefix or highly_similar or similar_same_source:
            duplicate_of = kept.get("clip_id", "")
            duplicate_similarity = round(similarity, 4)
            if same_exact_text:
                duplicate_reason = "exact_text_duplicate"
            elif same_prefix:
                duplicate_reason = "same_prefix_duplicate"
            elif similar_same_source:
                duplicate_reason = "same_source_near_duplicate"
            else:
                duplicate_reason = "high_similarity_duplicate"
            break

    current["dedup_duplicate_of"] = duplicate_of
    current["dedup_reason"] = duplicate_reason
    current["dedup_similarity"] = duplicate_similarity

    if duplicate_of:
        current["dedup_status"] = "dropped"
        dedup_review_rows.append(current)
    else:
        current["dedup_status"] = "kept"
        current["_dedup_norm_text"] = current_norm_text
        kept_rows.append(current)
        dedup_review_rows.append(dict(current))

final_deduped_rows = []
for row in kept_rows:
    clean_row = dict(row)
    clean_row.pop("_dedup_norm_text", None)
    clean_row["final_status"] = "approved_deduped"
    final_deduped_rows.append(clean_row)

final_dedup_review_path = OUTPUT_DIR / "final_dedup_review.jsonl"
final_dedup_summary_path = OUTPUT_DIR / "final_dedup_summary.json"

write_jsonl(dedup_review_rows, final_dedup_review_path)

final_dedup_summary = {
    "clips_before_dedup": len(qa_passed_item_rows),
    "clips_after_dedup": len(final_deduped_rows),
    "duplicates_removed": len(qa_passed_item_rows) - len(final_deduped_rows),
    "dedup_keep_rate": round(len(final_deduped_rows) / max(len(qa_passed_item_rows), 1), 3),
    "exact_text_duplicate": sum(1 for row in dedup_review_rows if row.get("dedup_reason") == "exact_text_duplicate"),
    "same_prefix_duplicate": sum(1 for row in dedup_review_rows if row.get("dedup_reason") == "same_prefix_duplicate"),
    "same_source_near_duplicate": sum(1 for row in dedup_review_rows if row.get("dedup_reason") == "same_source_near_duplicate"),
    "high_similarity_duplicate": sum(1 for row in dedup_review_rows if row.get("dedup_reason") == "high_similarity_duplicate"),
}

with open(final_dedup_summary_path, "wb") as f:
    f.write(orjson.dumps(final_dedup_summary, option=orjson.OPT_INDENT_2))

print("saved:", final_dedup_review_path)
print("saved:", final_dedup_summary_path)
print(json.dumps(final_dedup_summary, indent=2, ensure_ascii=False))
pd.DataFrame(final_deduped_rows)[["clip_id", "episode_id", "qa_score", "llm_score", "final_status"]].head(10)


In [ ]:
final_item_metadata_path = OUTPUT_DIR / "item_metadata_final.jsonl"
final_item_metadata_csv_path = OUTPUT_DIR / "item_metadata_final.csv"
final_dataset_summary_path = OUTPUT_DIR / "final_dataset_summary.json"

final_item_rows = []
for row in final_deduped_rows:
    final_row = dict(row)
    final_row["final_status"] = "approved_final"
    final_item_rows.append(final_row)

write_jsonl(final_item_rows, final_item_metadata_path)
pd.DataFrame(final_item_rows).to_csv(final_item_metadata_csv_path, index=False)

final_dataset_summary = {
    "episodes_total": len(episode_index),
    "clips_before_qa": len(item_metadata_rows),
    "clips_after_qa": len(qa_passed_item_rows),
    "clips_after_dedup": len(final_item_rows),
    "qa_rejected": len(item_metadata_rows) - len(qa_passed_item_rows),
    "dedup_removed": len(qa_passed_item_rows) - len(final_item_rows),
    "final_keep_rate_vs_qa_input": round(len(final_item_rows) / max(len(item_metadata_rows), 1), 3),
    "avg_final_llm_score": round(sum(float(row.get("llm_score", 0.0)) for row in final_item_rows) / max(len(final_item_rows), 1), 3),
    "avg_final_qa_score": round(sum(float(row.get("qa_score", 0.0)) for row in final_item_rows) / max(len(final_item_rows), 1), 3),
}

with open(final_dataset_summary_path, "wb") as f:
    f.write(orjson.dumps(final_dataset_summary, option=orjson.OPT_INDENT_2))

print("saved:", final_item_metadata_path)
print("saved:", final_item_metadata_csv_path)
print("saved:", final_dataset_summary_path)
print(json.dumps(final_dataset_summary, indent=2, ensure_ascii=False))
pd.DataFrame(final_item_rows)[["clip_id", "episode_id", "llm_score", "qa_score", "final_status"]].head(10)


## Dong Goi Dataset De Publish

Cell duoi day tao mot thu muc dataset sach de publish len Kaggle hoac tai su dung cho cac notebook sau.

Mac dinh se dong goi:
- `item_metadata_final.jsonl`
- `item_metadata_final.csv`
- `final_dataset_summary.json`
- `final_dedup_summary.json`
- `final_clip_qa_summary.json`
- `README.md`
- `dataset_manifest.json`

Neu muon dong goi ca audio clips cuoi cung, bat `COPY_FINAL_CLIPS = True` trong cell ben duoi.


In [ ]:
DATASET_VERSION_NAME = "ready_items_v1"
COPY_FINAL_CLIPS = False
PUBLISH_ROOT = WORK_DIR / "publish" / DATASET_VERSION_NAME
PUBLISH_ROOT.mkdir(parents=True, exist_ok=True)

files_to_copy = {
    final_item_metadata_path: PUBLISH_ROOT / "item_metadata_final.jsonl",
    final_item_metadata_csv_path: PUBLISH_ROOT / "item_metadata_final.csv",
    final_dataset_summary_path: PUBLISH_ROOT / "final_dataset_summary.json",
    final_dedup_summary_path: PUBLISH_ROOT / "final_dedup_summary.json",
    qa_summary_path: PUBLISH_ROOT / "final_clip_qa_summary.json",
}

copied_files = []
for src_path, dst_path in files_to_copy.items():
    if Path(src_path).exists():
        shutil.copy2(src_path, dst_path)
        copied_files.append(str(dst_path))

if COPY_FINAL_CLIPS:
    clips_publish_root = PUBLISH_ROOT / "clips"
    clips_publish_root.mkdir(parents=True, exist_ok=True)
    copied_clip_count = 0
    for row in final_item_rows:
        rel_clip_path = Path(row.get("clip_audio_path", ""))
        src_clip_path = WORK_DIR / rel_clip_path
        dst_clip_path = PUBLISH_ROOT / rel_clip_path.name if rel_clip_path.parent.name == "." else PUBLISH_ROOT / rel_clip_path
        dst_clip_path.parent.mkdir(parents=True, exist_ok=True)
        if src_clip_path.exists():
            shutil.copy2(src_clip_path, dst_clip_path)
            copied_clip_count += 1
else:
    copied_clip_count = 0

readme_text = f"""# PodTok Ready Items Dataset

Version: {DATASET_VERSION_NAME}

This packaged dataset was exported from the clip generation notebook.

Included files:
- item_metadata_final.jsonl
- item_metadata_final.csv
- final_dataset_summary.json
- final_dedup_summary.json
- final_clip_qa_summary.json
- dataset_manifest.json

Main file to use in downstream notebooks:
- item_metadata_final.jsonl

Notes:
- This package is the final dataset after QA and dedup.
- If COPY_FINAL_CLIPS was set to True, audio clips are also included.
""".strip()

readme_path = PUBLISH_ROOT / "README.md"
readme_path.write_text(readme_text, encoding="utf-8")

manifest = {
    "dataset_name": DATASET_VERSION_NAME,
    "main_file": "item_metadata_final.jsonl",
    "clips_included": COPY_FINAL_CLIPS,
    "copied_clip_count": copied_clip_count,
    "copied_files": [Path(path).name for path in copied_files] + ["README.md"],
    "episodes_total": len(episode_index),
    "clips_final": len(final_item_rows),
}
manifest_path = PUBLISH_ROOT / "dataset_manifest.json"
with open(manifest_path, "wb") as f:
    f.write(orjson.dumps(manifest, option=orjson.OPT_INDENT_2))

print("publish_root:", PUBLISH_ROOT)
print("copied_files:")
for path in copied_files:
    print(" -", path)
print("saved:", readme_path)
print("saved:", manifest_path)
print(json.dumps(manifest, indent=2, ensure_ascii=False))
